# Tarea 4: selección de rasgos mediante ranking de consenso

Notebook ejecutable para calcular siete métricas, sus rankings, el consenso Top-K, validación en prueba, estabilidad, visualizaciones, exportación y conclusiones. Cambia `DATA_PATH` y `TARGET` para usar otro CSV.

In [4]:
!pip install skrebate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.4/55.4 kB 3.2 MB/s eta 0:00:00


In [9]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.inspection import permutation_importance
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from skrebate import ReliefF
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone


warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
RANDOM_STATE, K, N_PERMUTATIONS = 42, 5, 20
DATA_PATH, TARGET = Path('/content/avp_1.csv'), 'Class'

In [6]:
if DATA_PATH.exists():
    datos = pd.read_csv(DATA_PATH)
    fuente = str(DATA_PATH)
    if TARGET not in datos.columns: TARGET = datos.columns[0]
else:
    xs, ys = make_classification(n_samples=600, n_features=18, n_informative=6, n_redundant=3, n_repeated=1, class_sep=1.2, random_state=RANDOM_STATE)
    datos = pd.DataFrame(xs, columns=[f'rasgo_{i+1:02d}' for i in range(xs.shape[1])]); datos[TARGET] = ys; fuente = 'datos sintéticos (respaldo)'
X = datos.drop(columns=TARGET).select_dtypes(include=np.number).replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median()).loc[:, lambda d: d.nunique() > 1]
y = datos[TARGET]

if not pd.api.types.is_numeric_dtype(y):
    y = pd.Series(pd.factorize(y)[0], index=y.index)

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.20, stratify=y, random_state=RANDOM_STATE)
print(f'Fuente: {fuente}\nDatos: {datos.shape}; X: {X.shape}; train: {X_train.shape}; test: {X_test.shape}')
display(pd.DataFrame({'tipo': datos.dtypes.astype(str), 'faltantes': datos.isna().sum(), 'unicos': datos.nunique()}))
display(y.value_counts().rename('frecuencia').to_frame().assign(proporcion=lambda d: d.frecuencia / len(y)))

Fuente: /content/avp_1.csv
Datos: (4908, 86); X: (4908, 85); train: (3926, 85); test: (982, 85)


,tipo,faltantes,unicos
Class,object,0,2
MIC_GOWAWA[0.0;1;NONE;0.0;0.0;2;W-OWA;0.4;0.5]_D_pbs,float64,0,381
GV[1]_GOWAWA[0.2;1;AO2-OWA;1.0;0.0;1;ES2-OWA;0.9;0.0]_C_cdch,float64,0,628
GV[1]_N2_T_pbs,float64,0,4899
GOWAWA[0.0;1;NONE;0.0;0.0;1;S-OWA;1.0;0.0]_T_mw,float64,0,19
...,...,...,...
TS[1]_N1_T_isa,float64,0,4818
AC[2]_N1_T_isa,float64,0,4901
AC[5]_N1_T_isa,float64,0,4849
TS[2]_P2_H_z3,float64,0,4304


,frecuencia,proporcion
0,2454,0.5
1,2454,0.5


## Métricas y rankings individuales

La selección usa únicamente entrenamiento. La permutación se mide en prueba como caída de `balanced_accuracy`, por lo que se reporta como evaluación de utilidad; para una práctica estricta puede sustituirse por una partición interna.

In [10]:

def normalized_numeric_entropy(x):
    max_entropy = np.log2(len(x))
    if max_entropy == 0:
        return 0.0
    counts, _ = np.histogram(x, bins=len(x))
    counts = counts[counts > 0]
    return float(stats.entropy(counts, base=2))/max_entropy if len(counts) and max_entropy else 0.0


def relief_score(x, y):
    z = StandardScaler().fit_transform(x)

    relief = ReliefF(
        n_neighbors=min(10, len(x) - 1),
        n_features_to_select=x.shape[1],
    )
    relief.fit(z, np.asarray(y))

    return pd.Series(
        relief.feature_importances_,
        index=x.columns,
        name="relieff",
    )


def mdi_score(x, y):
    rf = RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight="balanced",
    )
    rf.fit(x, y)

    return pd.Series(
        rf.feature_importances_,
        index=x.columns,
        name="mdi",
    )


def mdi_score_cv(x, y):
    modelo = RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight="balanced",
    )
    cv = StratifiedKFold(
        n_splits=10,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    y_array = np.asarray(y)
    importancias = []

    for train_idx, _ in cv.split(x, y_array):
        rf = clone(modelo)
        rf.fit(x.iloc[train_idx], y_array[train_idx])
        importancias.append(rf.feature_importances_)

    mdi_cv = pd.DataFrame(importancias, columns=x.columns)

    return pd.DataFrame({
        "MDI_medio_CV": mdi_cv.mean(),
        "MDI_std_CV": mdi_cv.std(),
    })


def permutation_score(X, y):

    rf = RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight="balanced",
    ).fit(X, y)

    per = permutation_importance(
        rf, X, y,
        scoring="balanced_accuracy",
        n_repeats=N_PERMUTATIONS,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    return pd.Series(
        per.importances_mean,
        index=X.columns,
        name="Permutación",
    )

def mutual_information_score(x, y):
    valores = mutual_info_classif(
        x, y,
        discrete_features=False,  # descriptores continuos
        n_neighbors=3,
        random_state=RANDOM_STATE,
    )
    return pd.Series(valores, index=x.columns, name="mi")

metric_values = pd.concat(
    [
        relief_score(X, y),
        mdi_score(X, y),
        mdi_score_cv(X, y),
        permutation_score(X, y),
        mutual_information_score(X, y)
    ],
    axis=1,
)

metric_values["entropy"] = X.apply(normalized_numeric_entropy)
metric_values['pearson'] = X.apply(lambda c: abs(stats.pearsonr(c, y)[0]) if c.nunique()>1 else 0)
metric_values['spearman'] = X.apply(lambda c: abs(stats.spearmanr(c, y).statistic) if c.nunique()>1 else 0)

    
metricas = ['pearson','spearman','mi','mdi','permutacion','relieff','entropy']

for m in metricas:
    metric_values[f'rank_{m}'] = metric_values[m].rank(method='average', ascending=False)

ranks = [f'rank_{m}' for m in metricas]
metric_values['rank_promedio'] = metric_values[ranks].mean(axis=1)
metric_values['rank_minimo'] = metric_values[ranks].min(axis=1)
metric_values['rank_maximo'] = metric_values[ranks].max(axis=1)
metric_values['rank_std'] = metric_values[ranks].std(axis=1)

ranking_final = metric_values.sort_values(['rank_promedio','rank_mi','rank_permutacion'], kind='mergesort').copy()
ranking_final['posicion_final'] = np.arange(1, len(ranking_final)+1)
ranking_final['seleccionado'] = ranking_final.posicion_final <= K
mejores_k = ranking_final.index[ranking_final.seleccionado].tolist()

display(ranking_final.head(K))

KeyError: 'permutacion'

## Visualizaciones

In [ ]:
# 1. Mapa de calor de los rankings
fig, ax = plt.subplots(
    figsize=(10, max(5, len(X.columns) * 0.24))
)

sns.heatmap(
    ranking_final[ranks],
    cmap="viridis_r",
    ax=ax,
    cbar_kws={"label": "Posición (1 = mejor)"},
)

ax.set(
    title="Mapa de calor de rankings",
    xlabel="Métrica",
    ylabel="Rasgo",
)

plt.tight_layout()
plt.show()


# 2. Los 15 rasgos con mejor ranking promedio
top15 = (
    ranking_final
    .head(min(15, len(ranking_final)))
    .sort_values("rank_promedio")
)

colores = [
    "#d62728" if seleccionado else "#4c78a8"
    for seleccionado in top15["seleccionado"]
]

fig, ax = plt.subplots(figsize=(9, 6))

ax.barh(
    top15.index.astype(str),
    top15["rank_promedio"],
    color=colores,
)

ax.invert_yaxis()
ax.set(
    title="Ranking promedio: mejores 15",
    xlabel="Posición promedio (menor es mejor)",
)

plt.tight_layout()
plt.show()


# 3. Comparación entre MDI e importancia por permutación
fig, ax = plt.subplots(figsize=(7, 6))

ax.scatter(
    ranking_final["rank_mdi"],
    ranking_final["rank_permutacion"],
    alpha=0.75,
)

diferencias = (
    ranking_final["rank_mdi"] - ranking_final["rank_permutacion"]
).abs()

discrepantes = diferencias.nlargest(
    min(5, len(ranking_final))
).index

for nombre in discrepantes:
    ax.annotate(
        str(nombre),
        (
            ranking_final.loc[nombre, "rank_mdi"],
            ranking_final.loc[nombre, "rank_permutacion"],
        ),
        fontsize=8,
    )

limite = max(
    ranking_final["rank_mdi"].max(),
    ranking_final["rank_permutacion"].max(),
)

ax.plot(
    [1, limite],
    [1, limite],
    "--",
    color="gray",
)

ax.set(
    title="MDI frente a permutación",
    xlabel="Rank MDI",
    ylabel="Rank permutación",
)

plt.tight_layout()
plt.show()

## Validación, estabilidad y conclusiones

In [ ]:
ranking_final.reset_index(names='rasgo').to_csv('ranking_consenso.csv', index=False); print('Exportado:', Path('ranking_consenso.csv').resolve()); display(ranking_final.head(K))